# 한 대화에서 배송 담당과 환불 담당 이어 주기

**먼저 `USE = "fake"`로 위에서부터 실행하세요. Solar 셀은 건너뛰세요.** 배송이 늦어진 주문의 위치를 묻고, 이어서 환불을 요청합니다. 사람이 환불을 승인한 뒤에는 “언제쯤 받을 수 있나요?”라고 묻습니다. 주문과 돈은 모두 노트북 안의 실습 자료입니다.

| 새 용어 | 여기서 뜻하는 것 |
|---|---|
| 라우터(router), 슈퍼바이저(supervisor) | 고객의 말과 대화 상태를 보고 다음 담당자를 고르는 단계입니다. |
| 담당 에이전트(specialist) | 배송이나 환불처럼 정해진 업무와 도구를 맡습니다. |
| 인계(handoff) | 진행 중인 대화의 담당이 바뀌는 일입니다. 출발·도착 담당과 이유를 남깁니다. |
| 서브그래프(subgraph), 노드 래퍼(node wrapper) | 서브그래프는 큰 그래프 안에서 실행하는 작은 그래프입니다. 래퍼는 안팎의 입력과 출력을 맞춰 주는 함수입니다. |
| 턴(turn) | 새 고객 메시지 하나로 시작하는 상담 단위입니다. 환불 승인으로 재개하면 같은 턴을 이어 갑니다. |
| `tasks` | 스냅샷의 실행 작업 목록입니다. 작업 이름과 그 작업에서 발생한 중단을 볼 수 있습니다. |

메시지, 체크포인터, `thread_id`, 도구 호출, 미들웨어, HITL의 기본 설명은 [대화 기억과 사람의 승인](./02_대화_기억과_사람의_승인.ipynb)에 있습니다. 이 노트북에서는 같은 대화 이름표를 쓰면서 **누가 이어 받고 어떤 메시지를 보는지** 살펴봅니다. 메모리 저장소는 커널을 재시작하면 비워집니다.

## 준비

프로젝트 경로를 맞추고 현재 커널의 파이썬과 라이브러리 버전을 출력합니다. `show()`는 메시지 종류와 내용, 도구 요청과 도구 결과의 이름·상태를 보여 줍니다. `thread_config()`는 새 대화 이름표를 만듭니다. 이 이름표는 한 번 만들어 배송 문의부터 환불 뒤 후속 질문까지 계속 씁니다.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from importlib.metadata import version
from typing import Annotated, Literal, TypedDict
import re
from uuid import uuid4

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, ModelRequest, dynamic_prompt
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package}: {version(package)}")

def show(messages):
    for message in messages:
        line = f"{message.type}: {message.content}"
        if isinstance(message, AIMessage) and message.tool_calls:
            line += " | tool_calls=" + json.dumps(message.tool_calls, ensure_ascii=False)
        if isinstance(message, ToolMessage):
            line += f" | name={message.name} status={message.status}"
        print(line.replace("\n", " ↵ "))

def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma 연결 준비

`LocalLLM(port=18380).client`로 shopagent와 같은 로컬 연결 객체를 만듭니다. `/health`를 최대 2초 동안 확인하고 HTTP 상태 코드나 연결 실패 안내를 출력합니다. 모델 질문은 보내지 않습니다. 서버가 꺼져 있어도 가짜 모델 실습을 계속할 수 있습니다.

나중에 Gemma를 써 볼 때는 별도 터미널에서 다음 명령으로 서버를 시작할 수 있습니다. 서버를 종료하려면 그 터미널에서 Ctrl+C를 누르세요.

```bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
```

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
print("Gemma 연결 객체 준비 완료. 모델 호출은 하지 않았습니다.")
try:
    with httpx.Client(trust_env=False) as health_client:
        health_response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    print(f"llama-server /health (상태 코드: {health_response.status_code})")
except httpx.HTTPError:
    print("llama-server 연결 실패. 가짜 모델로 계속할 수 있습니다.")

### Solar 연결 준비

가짜 모델 실습에서는 이 셀을 건너뛰세요. Solar에 실제로 질문하면 **Upstage API 사용 비용**이 듭니다. 이 셀은 연결만 준비합니다.

앞 노트북과 같이 bash의 `.` 명령으로만 키 설정을 불러옵니다. 셸 출력을 내부에서 받아 키가 있으면 환경 변수에 넣습니다. 화면에는 로드 여부(True/False)만 표시합니다. 키를 준비하지 못하면 `solar_llm`은 `None`이고 가짜 모델로 진행하라는 안내가 나옵니다. 키, 환경 변수 전체, 셸 출력과 오류는 화면에 표시하지 마세요.

In [ ]:
from shop.servers import CONFIGS

def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO, capture_output=True, text=True, timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())

solar_loaded = load_solar_key()
print("Upstage 키 loaded:", solar_loaded)
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
if solar_llm is None:
    print("키가 없어 Solar를 준비하지 못했습니다. USE = 'fake'로 진행하세요.")

### 가짜 모델 규칙

`BaseChatModel`을 상속합니다. `bind_tools()`는 그 담당이 받은 도구 이름을 복사본에 붙입니다. 모델은 시스템 지시의 마지막 줄에서 주문 번호를 읽습니다. 주문 번호가 없다면 고객에게 번호를 묻습니다.

배송 담당은 `track_order`를 요청합니다. 환불 담당은 “환불해”나 “환불 처리해” 같은 표현이 들어오면 `issue_refund`를, 그 밖의 후속 질문에는 `refund_status`를 요청합니다. 도구가 답을 주면 그 내용을 고객 문장으로 돌려줍니다. 사람이 거절하면 실행하지 않았다고 답합니다. 이 셀은 가짜 모델의 준비 상태만 출력합니다.

In [ ]:
ORDER_NUMBER = re.compile(r"(?<![A-Za-z0-9])O[0-9]+(?![A-Za-z0-9])")

class ScriptedSpecialistModel(BaseChatModel):
    tool_names: tuple[str, ...] = ()

    @property
    def _llm_type(self):
        return "scripted-specialist"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self.model_copy(update={"tool_names": tuple(tool.name for tool in tools)})

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        last = messages[-1]
        if isinstance(last, ToolMessage):
            text = ("환불을 실행하지 않았습니다." if last.status == "error"
                    else json.loads(last.content)["message"])
            reply = AIMessage(content=text)
        else:
            customer = next(message for message in reversed(messages) if isinstance(message, HumanMessage))
            instructions = next(message for message in messages if isinstance(message, SystemMessage))
            order = ORDER_NUMBER.search(instructions.content.splitlines()[-1])
            if order is None:
                reply = AIMessage(content="주문 번호를 알려 주세요.")
            else:
                text = "".join(customer.content.split())
                requests_refund = any(phrase in text for phrase in ("환불해", "환불처리해", "환불을해"))
                if "track_order" in self.tool_names:
                    name = "track_order"
                else:
                    name = "issue_refund" if requests_refund else "refund_status"
                reply = AIMessage(content="", tool_calls=[{
                    "name": name,
                    "args": {"order_id": order.group()},
                    "id": f"{name}-{customer.id}",
                    "type": "tool_call",
                }])
        return ChatResult(generations=[ChatGeneration(message=reply)])

fake_llm = ScriptedSpecialistModel()
print("가짜 모델 준비 완료. 규칙으로 답합니다.")

### 모델 선택

`선택` 줄에 이후 셀에서 쓸 모델이 나옵니다. 모델을 바꾸면 이 셀부터 다시 실행하세요. Gemma는 위의 서버 상태를 먼저 확인하세요. Solar 셀을 건너뛰고 Solar를 고르면 연결을 준비하라는 안내가 나옵니다. 실제 모델의 답변과 도구 선택은 달라질 수 있습니다.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"
models = {"fake": fake_llm, "gemma": globals().get("gemma_llm"), "solar": globals().get("solar_llm")}
assert USE in models, "USE는 fake, gemma, solar 중 하나여야 합니다."
llm = models[USE]
assert llm is not None, "선택한 모델이 준비되지 않았습니다. 해당 준비 셀을 실행하거나 USE = 'fake'로 바꾸세요."
print("선택:", USE)
if USE == "gemma":
    print("위 Gemma 셀의 서버 상태 검사 결과를 확인하세요. 서버가 꺼져 있다면 위 터미널 명령으로 실행한 뒤 진행하세요.")

## 1. 배송 조회와 환불 도구

O701은 배송 지연, O702는 분실 확인, O703은 정상 배송 중입니다. 배송 담당은 위치만 조회할 수 있습니다. 환불 담당은 환불을 실행하거나 처리 상태를 조회할 수 있습니다.

`issue_refund`는 배송 지연·분실 확인 주문만 환불 완료로 바꿉니다. 실습에서는 이 변경을 되돌리는 도구가 없습니다. 이미 환불한 주문에 다시 실행하면 기존 완료 상태를 안내합니다. `refund_runs`에는 실제로 환불 상태를 바꾼 번호만 추가합니다. 도구 결과의 `ok`는 업무 결과이고, `ToolMessage.status`는 도구 응답의 표식입니다. 승인을 받아 도구가 응답해도 배송 중이면 업무 결과는 `ok: false`입니다.

자료는 딕셔너리 안에 있으므로 대화 스냅샷과 수명이 따로입니다. 다음 비교 실습은 주문 딕셔너리도 새로 만듭니다.

In [ ]:
ORDER_SEED = {
    "O701": {"delivery_status": "배송 지연", "last_scan": "대전 물류센터", "amount": 32900, "refunded": False},
    "O702": {"delivery_status": "분실 확인", "last_scan": "부산 집하점", "amount": 21500, "refunded": False},
    "O703": {"delivery_status": "배송 중", "last_scan": "서울 배송점", "amount": 18400, "refunded": False},
}

def new_orders():
    return {order_id: data.copy() for order_id, data in ORDER_SEED.items()}

def make_shop_tools(orders, refund_runs):
    @tool
    def track_order(order_id: Annotated[str, "배송을 확인할 주문 번호"]) -> dict:
        """운송장에 남은 주문의 배송 상태와 마지막 위치를 조회합니다."""
        order = orders.get(order_id)
        if order is None:
            return {"ok": False, "message": f"{order_id} 주문을 찾을 수 없습니다."}
        status, location = order["delivery_status"], order["last_scan"]
        return {"ok": True, "message": f"{order_id}은(는) {status} 상태입니다. 마지막 위치는 {location}입니다."}

    @tool
    def issue_refund(order_id: Annotated[str, "되돌릴 수 없는 환불을 실행할 주문 번호"]) -> dict:
        """승인 후 배송 지연 또는 분실이 확인된 주문을 환불합니다."""
        order = orders.get(order_id)
        if order is None:
            return {"ok": False, "message": f"{order_id} 주문을 찾을 수 없습니다."}
        if order["refunded"]:
            return {"ok": True, "message": f"{order_id}은(는) 이미 환불되었습니다."}
        if order["delivery_status"] not in ("배송 지연", "분실 확인"):
            return {"ok": False, "message": f"{order_id}은(는) 배송 중이라 환불하지 않았습니다."}
        order["refunded"] = True
        refund_runs.append(order_id)
        return {"ok": True, "message": f"{order_id} 주문의 {order['amount']:,}원 환불을 처리했습니다."}

    @tool
    def refund_status(order_id: Annotated[str, "환불 상태를 확인할 주문 번호"]) -> dict:
        """주문의 환불 상태와 입금 예정 기간을 조회합니다."""
        order = orders.get(order_id)
        if order is None:
            return {"ok": False, "message": f"{order_id} 주문을 찾을 수 없습니다."}
        if not order["refunded"]:
            return {"ok": True, "message": f"{order_id}에 완료된 환불이 없어 입금 예정일이 없습니다."}
        return {"ok": True, "message": f"{order_id} 환불은 완료됐습니다. 입금은 영업일 기준 3~5일이 걸립니다."}

    return track_order, issue_refund, refund_status

## 2. 라우터가 담당을 정하고 인계를 남기기

`MessagesState`를 확장해 `active_specialist`(현재 담당), `order_id`(상담 중인 주문), `route_reason`(이번 선택 이유), `handoffs`(담당 변경 기록)를 함께 저장합니다. 매 턴 라우터가 마지막 고객 메시지를 확인합니다.

1. “환불”, “입금”이 있으면 환불 담당으로 보냅니다. 배송과 환불이 함께 나오면 환불을 먼저 다룹니다.
2. “배송”, “택배”, “운송장”이 있으면 배송 담당으로 보냅니다.
3. 단서가 없으면 지난 담당을 유지합니다. 첫 문의부터 단서가 없다면 배송 담당이 받습니다.

이전 담당과 달라질 때만 인계를 기록합니다. 주문 번호가 새로 나오면 그 번호를 쓰고, 없으면 저장된 번호를 이어 씁니다. `Command(update=..., goto=...)`는 상태를 갱신하고 담당 노드로 보냅니다. 같은 선택을 늘 재현할 수 있지만, 단어만 보는 라우터는 “환불은 됐고요, 배송만 알려 주세요”처럼 복합적인 문장에 약합니다.

In [ ]:
Specialist = Literal["delivery", "refund"]

class Handoff(TypedDict):
    from_specialist: Specialist
    to_specialist: Specialist
    reason: str

class ConversationState(MessagesState):
    active_specialist: Specialist
    order_id: str | None
    route_reason: str
    handoffs: list[Handoff]

def current_turn(messages):
    last_customer = next(i for i in range(len(messages) - 1, -1, -1)
                         if isinstance(messages[i], HumanMessage))
    return messages[last_customer:]

def route_specialist(state: ConversationState) -> Command[Specialist]:
    customer_text = current_turn(state["messages"])[0].content
    previous = state.get("active_specialist")
    if any(word in customer_text for word in ("환불", "입금")):
        specialist, reason = "refund", "환불·입금 단서"
    elif any(word in customer_text for word in ("배송", "택배", "운송장")):
        specialist, reason = "delivery", "배송 단서"
    elif previous:
        specialist, reason = previous, "새 단서가 없어 이전 담당 유지"
    else:
        specialist, reason = "delivery", "첫 문의의 기본 담당"

    mentioned_order = ORDER_NUMBER.search(customer_text)
    order_id = mentioned_order.group() if mentioned_order else state.get("order_id")
    handoffs = state.get("handoffs", [])
    if previous is not None and previous != specialist:
        handoffs = [*handoffs, {"from_specialist": previous, "to_specialist": specialist, "reason": reason}]
    return Command(update={
        "active_specialist": specialist,
        "order_id": order_id,
        "route_reason": reason,
        "handoffs": handoffs,
    }, goto=specialist)

### 공유 기록에서 담당에게 보낼 범위 고르기

`START → router → delivery 또는 refund → END` 순서입니다. 두 담당은 `create_agent`로 만든 서브그래프입니다. 래퍼에서 공유 기록을 입력으로 바꾸고, 담당이 새로 만든 메시지만 바깥 기록에 붙입니다. 부모에 붙인 `InMemorySaver` 하나를 두 담당도 이어받습니다.

- **`full`**은 앞선 질문, 답변, 도구 호출과 결과까지 모두 넘깁니다. 배송 근거를 다시 읽을 수 있지만 대화가 길어지고 다른 업무의 도구 결과도 보입니다.
- **`current_turn`**은 마지막 고객 말부터 넘깁니다. 이전 도구 호출과 결과를 한 쌍으로 함께 제외합니다. 앞 대화의 상세 내용은 줄고, “그럼”이 가리키는 내용을 일부 놓칠 수 있습니다.
- 두 방식 모두 `order_id`를 별도로 넘깁니다. `dynamic_prompt`가 그 값을 담당의 시스템 지시에 넣습니다. 그래서 지난 메시지를 빼도 “그럼 환불해 주세요”의 주문을 찾을 수 있습니다. 환불 가능 여부는 도구가 주문 자료에서 다시 확인합니다.

환불 담당의 HITL 미들웨어는 `issue_refund` 호출 직전에 `interrupt()`를 사용합니다. 확인 문구에는 주문, 배송 상태, 금액이 나옵니다. 래퍼 안에서 중단이 전파되어 바깥 그래프도 멈춥니다. 재개 때 래퍼가 다시 시작될 수 있으므로, 래퍼에서는 입력을 준비하고 결과를 합치는 일만 합니다.

In [ ]:
SPECIALIST_PROMPTS = {
    "delivery": (
        "배송 담당입니다. 고객에게 한국어로 답하세요. 주문 번호가 있으면 track_order로 배송을 조회하세요."
    ),
    "refund": (
        "환불 담당입니다. 고객에게 한국어로 답하세요. 환불 실행 요청이면 issue_refund를, "
        "입금이나 처리 상태 문의면 refund_status를 호출하세요. issue_refund의 승인은 프로그램이 받습니다."
    ),
}

class SpecialistContext(TypedDict):
    specialist: Specialist
    order_id: str | None

def prompt_text(specialist, order_id):
    return (
        f"{SPECIALIST_PROMPTS[specialist]}\n"
        "주문 번호가 없으면 번호를 물어보세요. 도구 결과를 받은 뒤에는 다시 호출하지 말고 그 결과를 안내하세요.\n"
        f"현재 상담 주문 번호: {order_id or '미확인'}"
    )

@dynamic_prompt
def specialist_prompt(request: ModelRequest) -> str:
    context = request.runtime.context
    return prompt_text(context["specialist"], context["order_id"])

def specialist_messages(messages, history_mode):
    return messages if history_mode == "full" else current_turn(messages)

def wrap_specialist(agent, history_mode):
    def run(state: ConversationState, config: RunnableConfig):
        visible = specialist_messages(state["messages"], history_mode)
        result = agent.invoke({"messages": visible}, config, context={
            "specialist": state["active_specialist"],
            "order_id": state["order_id"],
        })
        return {"messages": result["messages"][len(visible):]}
    return run

def build_handoff_graph(model, orders, refund_runs, *, history_mode="full"):
    if history_mode not in ("full", "current_turn"):
        raise ValueError("history_mode는 full 또는 current_turn이어야 합니다.")
    track_order, issue_refund, refund_status = make_shop_tools(orders, refund_runs)

    def confirmation_card(call, state, runtime):
        order_id = call["args"]["order_id"]
        order = orders.get(order_id)
        details = (f"{order['delivery_status']}, {order['amount']:,}원"
                   if order else "등록된 주문 정보 없음")
        return f"{order_id} 환불을 실행할까요? ({details})"

    delivery_agent = create_agent(model, [track_order], context_schema=SpecialistContext,
                                  middleware=[specialist_prompt], name="delivery")
    refund_agent = create_agent(
        model, [issue_refund, refund_status], context_schema=SpecialistContext, name="refund",
        middleware=[
            specialist_prompt,
            HumanInTheLoopMiddleware(
                interrupt_mode="per_call",
                interrupt_on={"issue_refund": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": confirmation_card,
                }},
            ),
        ],
    )

    builder = StateGraph(ConversationState)
    builder.add_node("router", route_specialist, destinations=("delivery", "refund"))
    builder.add_node("delivery", wrap_specialist(delivery_agent, history_mode))
    builder.add_node("refund", wrap_specialist(refund_agent, history_mode))
    builder.add_edge(START, "router")
    builder.add_edge("delivery", END)
    builder.add_edge("refund", END)
    return builder.compile(checkpointer=InMemorySaver())

### 어디까지 실행됐는지 읽는 도우미

`get_state_history()`의 기록을 최신 순으로 읽어, 이번 턴에서 라우터 직전과 담당 직전의 상태를 찾습니다. `get_state(subgraphs=True)`는 현재 바깥 상태와 대기 중인 안쪽 작업의 상태를 펼쳐 줍니다.

- `상태`에는 현재 담당, 주문 번호, 선택 이유, 인계 이력과 메시지 종류가 나옵니다. 담당 내부 상태에는 그 에이전트가 가진 메시지만 표시합니다.
- `next`는 이어서 실행할 노드입니다. `tasks`는 각 작업의 이름과 중단 수를 보여 줍니다. 과거 스냅샷은 그 시점에 예정돼 있던 작업과, 이후 그 작업이 일으킨 중단도 보여 줍니다. 현재 스냅샷에서 둘 다 `()`이면 작업이 끝났습니다.
- `담당의 첫 모델 입력`에는 담당 직전 스냅샷으로 다시 만든 시스템 지시와 실제 전달 범위가 나옵니다. 도구가 응답하면 그 턴의 도구 요청·결과도 다음 모델 호출에 추가됩니다.
- `공유 기록의 이번 턴`은 바깥에 합쳐진 메시지입니다. 환불이 멈춰 있으면 안쪽의 요청은 아직 여기에 없어서, 안쪽 작업의 마지막 메시지를 따로 봅니다. `interrupt.id`는 사람의 응답을 연결할 이름표입니다. `interrupt.value`는 어떤 도구와 주문에 대해 무엇을 확인하는지 보여 줍니다.

새 고객 메시지는 대기가 끝난 뒤에 받습니다. 승인 대기 중이라면 먼저 현재 중단에 대한 결정을 보내세요.

In [ ]:
def show_snapshot(label, snapshot, *, parent=True):
    values = snapshot.values
    print(label)
    fields = {
        field: values.get(field)
        for field in ("active_specialist", "order_id", "route_reason", "handoffs")
    } if parent else {}
    fields["messages"] = [message.type for message in values.get("messages", [])]
    print("  상태:", fields)
    print("  next:", snapshot.next,
          "| tasks (이름, 중단 수):", tuple((task.name, len(task.interrupts)) for task in snapshot.tasks))

def inspect_current(graph, config):
    snapshot = graph.get_state(config, subgraphs=True)
    show_snapshot("현재 바깥 상태", snapshot)
    print("공유 기록의 이번 턴")
    show(current_turn(snapshot.values["messages"]))
    for task in snapshot.tasks:
        if task.interrupts and task.state is not None:
            show_snapshot(f"대기 중인 {task.name} 내부 상태", task.state, parent=False)
            print("담당 내부의 마지막 메시지")
            show(task.state.values["messages"][-1:])
    for pending in snapshot.interrupts:
        print("interrupt.id:", pending.id)
        print("interrupt.value:", json.dumps(pending.value, ensure_ascii=False))
    return snapshot

def inspect_turn(graph, config, history_mode):
    history = list(graph.get_state_history(config))
    before = next(saved for saved in history if saved.next == ("router",))
    after = next(saved for saved in history if saved.next == (saved.values.get("active_specialist"),))
    show_snapshot("라우터 실행 전", before)
    show_snapshot("라우터 선택 후", after)
    print("담당의 첫 모델 입력")
    state = after.values
    prompt = SystemMessage(content=prompt_text(state["active_specialist"], state["order_id"]))
    show([prompt, *specialist_messages(state["messages"], history_mode)])
    return inspect_current(graph, config)

def send_turn(graph, config, text):
    snapshot = graph.get_state(config)
    if snapshot.interrupts:
        raise RuntimeError("환불 승인 대기 중입니다. 먼저 현재 중단에 답하세요.")
    customers = sum(isinstance(message, HumanMessage) for message in snapshot.values.get("messages", []))
    graph.invoke({"messages": [HumanMessage(content=text, id=f"customer-{customers + 1}")]}, config)

## 3. 한 대화에서 배송 → 환불 → 환불 후속 질문

전체 메시지를 넘기는 `full`부터 시작합니다. 아래 `대화` 줄에는 새 이름표와 전달 범위가 나옵니다. 뒤의 세 요청과 재개는 이 그래프와 `config`를 함께 씁니다.

In [ ]:
history_mode = "full"  # "full" | "current_turn"
orders = new_orders()
refund_runs = []
graph = build_handoff_graph(llm, orders, refund_runs, history_mode=history_mode)
config = thread_config("specialist")
print("대화:", config["configurable"]["thread_id"], "| 전달 범위:", history_mode)

### 첫 질문은 배송 담당에게

라우터 실행 전에는 `active_specialist`가 비어 있습니다. 라우터가 배송 단서와 O701을 찾으면 `next`에 `delivery`가 나옵니다. 첫 배정이라 인계 목록은 비어 있습니다. 배송 담당은 고객 말과 시스템 지시를 받고 위치를 조회합니다. 현재 상태의 `next`와 `tasks`는 빈 튜플이고, 이번 턴 기록에는 고객 말, 조회 요청, 조회 결과, 배송 안내가 차례로 보입니다.

In [ ]:
delivery_question = "주문 O701 배송이 늦어요. 지금 어디쯤인가요?"
send_turn(graph, config, delivery_question)
delivery_snapshot = inspect_turn(graph, config, history_mode)

### 두 번째 질문은 환불 담당에게

“그럼 환불해 주세요”에는 주문 번호가 없습니다. 저장된 O701을 쓰고 `delivery → refund` 인계를 기록합니다. `full`에서 환불 담당의 첫 입력에 대전 물류센터 조회 결과까지 들어오는지 보세요.

가짜 모델은 `issue_refund`를 요청합니다. 바깥 `next`에는 `refund`가 남고, 안쪽 `next`에는 도구 작업(`tools`)이 남습니다. 안쪽 마지막 메시지에서 환불 요청을 확인할 수 있습니다. `interrupt.value`의 확인 문구에는 O701과 32,900원이 보입니다. `환불 완료 여부`는 이 주문의 딕셔너리 값이고, `환불 처리 기록`은 상태를 바꾼 주문 목록입니다. 승인 전에는 각각 `False`와 `[]`입니다. 실제 모델이 도구를 요청하지 않으면 중단 정보 대신 그 답변이 보입니다.

In [ ]:
def show_refund_record(snapshot, orders, refund_runs):
    order_id = snapshot.values["order_id"]
    print("환불 완료 여부:", {order_id: orders.get(order_id, {}).get("refunded")})
    print("환불 처리 기록:", refund_runs)

refund_request = "그럼 환불해 주세요."
send_turn(graph, config, refund_request)
paused_snapshot = inspect_turn(graph, config, history_mode)
show_refund_record(paused_snapshot, orders, refund_runs)

### 같은 대화에서 승인하고 이어 가기

`Command(resume={중단 ID: 결정})`을 **같은 그래프와 같은 `config`**에 보냅니다. 새 고객 메시지를 넣지 않으므로 라우터를 다시 선택하지 않고 대기 중이던 환불 작업을 이어 갑니다. 실제 화면에서는 사람이 버튼으로 고를 결정을 여기서는 딕셔너리로 넣습니다. 이전 메시지에 붙는 환불 요청, 도구 결과, 담당 답변을 확인하세요.

기본 예제에서는 `next`와 `tasks`가 비고 O701 완료 여부가 `True`가 됩니다. 처리 기록은 `['O701']`이고 인계는 한 건 그대로입니다. 중단이 없었던 실모델 실행에서는 재개를 건너뛰었다는 안내가 나옵니다.

In [ ]:
def resume_refund(graph, config, decision):
    snapshot = graph.get_state(config)
    if not snapshot.interrupts:
        print("재개할 승인 대기가 없습니다.")
        return snapshot
    pending = snapshot.interrupts[0]
    graph.invoke(Command(resume={pending.id: decision}), config)
    return inspect_current(graph, config)

decision = {"type": "approve"}
resumed_snapshot = resume_refund(graph, config, decision)
show_refund_record(resumed_snapshot, orders, refund_runs)

### 다음 고객 말은 새 턴, 담당은 그대로

“언제쯤 받을 수 있나요?”에는 배송·환불 단서도 주문 번호도 없습니다. 라우터는 저장된 `refund`와 O701을 씁니다. 첫 입력에서 배송과 환불 대화가 모두 이어지는지 보세요. 가짜 모델은 `refund_status`를 조회하고 영업일 기준 입금 기간을 안내합니다. 조회이므로 `next`와 `tasks`는 비고, 인계와 환불 처리 기록은 그대로입니다.

In [ ]:
followup_config = config
followup_question = "언제쯤 받을 수 있나요?"
send_turn(graph, followup_config, followup_question)
followup_snapshot = inspect_turn(graph, followup_config, history_mode)
show_refund_record(followup_snapshot, orders, refund_runs)

## 4. 이전 배송 기록을 빼고 같은 인계 해 보기

별도의 실습 그래프와 주문 딕셔너리에 같은 두 질문을 보냅니다. 여기서는 가짜 모델을 고정해서 전달 범위 차이를 같은 규칙으로 비교합니다.

- `라우터 선택 후`의 바깥 상태에는 첫 질문과 배송 조회·안내가 저장돼 있습니다.
- `담당의 첫 모델 입력`에는 시스템 지시와 “그럼 환불해 주세요”만 보입니다. 시스템 지시에 O701이 남아 있으므로 같은 주문의 환불 승인을 요청합니다.
- `대기 중인 refund 내부 상태`에는 이번 고객 말과 방금 만든 도구 요청이 보입니다. 이전 배송 메시지는 바깥 상태에서 계속 읽을 수 있습니다.
- 이 비교에서는 `reject`로 마칩니다. 도구 메시지에 거절 사유가 붙고 `status=error`가 보입니다. 이는 사람이 거절했다는 표시입니다. 환불 완료 여부는 `False`, 처리 기록은 `[]`입니다.

In [ ]:
filtered_orders = new_orders()
filtered_runs = []
filtered_graph = build_handoff_graph(fake_llm, filtered_orders, filtered_runs, history_mode="current_turn")
filtered_config = thread_config("filtered")
send_turn(filtered_graph, filtered_config, "주문 O701 배송이 늦어요. 지금 어디쯤인가요?")
send_turn(filtered_graph, filtered_config, "그럼 환불해 주세요.")
filtered_paused = inspect_turn(filtered_graph, filtered_config, "current_turn")
filtered_resumed = resume_refund(filtered_graph, filtered_config, {
    "type": "reject", "message": "배송을 조금 더 기다릴게요.",
})
show_refund_record(filtered_resumed, filtered_orders, filtered_runs)

## 5. 이 구조를 고른 이유

| 설계 선택 | 실습에서 보이는 이점과 부담 |
|---|---|
| 규칙 라우터와 도구로 인계하기 | 규칙 라우터를 쓰면 담당 선택이 한 곳에 모여 있고 같은 입력을 재현하기 쉽습니다. 담당이 도구로 라우터에 돌려보내게 하면 상담 도중 새로 드러난 의도를 반영할 수 있습니다. 그 경우 도구 호출과 응답을 짝지어 기록하고 인계 횟수도 관리해야 합니다. |
| 공유 메시지와 담당별 메시지 | 공유 목록이 하나면 상담 전후를 이어 보거나 검토하기 쉽습니다. 담당별로 목록을 나누면 업무별 맥락 범위가 분명해집니다. 그 경우 주문 번호, 요약, 진행 상태처럼 상대 담당에 필요한 사실을 따로 넘기는 규칙이 필요합니다. 여기서는 공유 목록에서 실제로 보낼 범위만 고릅니다. |
| 서브그래프 직접 연결과 래퍼 | 상태가 같으면 컴파일된 담당 그래프를 노드로 바로 연결할 수 있습니다. 여기서는 래퍼가 부모 상태를 담당 입력으로 바꾸고 새 메시지만 부모로 돌려보냅니다. 입력과 출력의 경계가 분명한 대신 래퍼가 어디서 다시 시작되는지까지 살펴야 합니다. 이 래퍼는 담당이 메시지를 뒤에 추가한다는 전제로 새 부분만 붙입니다. 담당이 이전 메시지를 수정하는 구조라면 그 변경도 반영해야 합니다. 환불 요청이 부모 기록에 합쳐지기 전에는 안쪽 스냅샷에서 확인할 수 있습니다. |

## 6. 직접 바꿔 볼 것

가짜 모델을 선택하세요. 실험마다 `start-thread`의 준비 셀부터 다시 실행해, 앞 실험에서 바뀐 주문과 대화가 섞이지 않게 하세요. 각 실험에서 한 가지 변경만 적용한 뒤 원래 값으로 돌려놓으세요.

1. **분실 주문 확인.** `delivery-turn`의 `delivery_question`에서 O701을 O702로 바꾸고 `resume-refund`까지 실행하세요. 위치는 부산 집하점이고, 승인 카드에는 21,500원이 나옵니다. 승인 후 처리 기록에 O702가 들어갑니다.
2. **주문 번호를 바꾼 환불.** `refund-turn`의 `refund_request`를 “주문 O703 환불해 주세요.”로 바꾸고 `resume-refund`까지 실행하세요. 확인 카드는 O703을 가리킵니다. 승인해도 현재 배송 중이므로 도구 결과의 `ok`는 `false`, 처리 기록은 `[]`입니다.
3. **승인 대신 거절.** `resume-refund`의 `decision`을 `{"type": "reject", "message": "조금 더 기다릴게요."}`로 바꾸고 `followup`까지 실행하세요. 환불 실행 없이 중단이 끝납니다. 다음 질문에는 환불 담당이 입금 예정일이 없다고 안내합니다.
4. **배송 담당으로 돌아가기.** `followup`의 `followup_question`을 “배송 위치를 다시 알려 주세요.”로 바꾸세요. 주문 번호 없이도 O701을 조회합니다. 상태에 `refund → delivery` 인계가 한 건 추가됩니다.
5. **다른 대화 이름표.** `followup`의 `followup_config`를 `thread_config("새문의")`로 바꾸세요. 같은 질문이 첫 문의가 되어 배송 담당에게 갑니다. 저장된 주문이 없어서 번호를 묻고, 인계 목록이 비어 있습니다.
6. **같은 상담, 짧은 입력.** `start-thread`의 `history_mode`를 `"current_turn"`으로 바꾸고 `refund-turn`까지 실행하세요. 바깥 메시지 종류와 인계는 그대로입니다. 첫 모델 입력에서는 배송 조회 결과가 빠지고 확인 카드는 여전히 O701을 가리킵니다. 이어서 `resume-refund`를 실행해 보세요.
7. **한 문장에 배송과 환불.** `refund-turn`의 `refund_request`를 “배송도 확인하고 환불해 주세요.”로 바꾸고 `resume-refund`까지 실행하세요. 환불 단서를 먼저 검사하므로 `next`에 `refund`가 보이고 배송 조회는 한 번 더 실행되지 않습니다.

### shopagent 코드에서 이어 읽기

| 여기서 본 것 | 프로젝트 위치와 역할 |
|---|---|
| 담당 선택과 인계 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_graph` 안 `router`는 `agent`를 갱신하고 `Command(goto=...)`로 보냅니다. [shop/agents.py](../shop/agents.py)의 `Agents.supervise`는 LLM을 쓸 때 `transfer_to_…` 호출로 담당을 고릅니다. |
| 담당에서 라우터로 복귀 | [shop/graph_v2.py](../shop/graph_v2.py)의 `TurnMiddleware._tool_call`은 `transfer_back_to_router`를 기록하고, `build_graph` 안 `after_agent`가 라우터 복귀를 결정합니다. |
| 공유 메시지와 담당 래퍼 | [shop/graph_v2.py](../shop/graph_v2.py)의 `State`, `build_graph` 안 `specialist`는 담당에 공유 메시지를 넘기고 결과를 부모 상태에 반영합니다. |
| 도구 승인과 대화 재개 | [shop/graph_v2.py](../shop/graph_v2.py)의 `build_agent`는 되돌리기 힘든 도구에 HITL을 붙입니다. `ScenarioGraph.__init__`은 저장소와 대화 이름표를 만들고, `resume_turn`이 중단 ID에 결정을 연결합니다. |